# Preprocessing et Feature Engineering

In [19]:
import pandas as pd
import numpy as np

## Date conversion and Data loading

In [20]:
df = pd.read_csv("../data/merged_dataset.csv")
df["delivery_time"] = pd.to_datetime(df['delivery_time'])

## Handling Missing Values

Linear interpolation for weather, as it's a continuous time series

In [21]:
df = df.groupby('site_name', group_keys=False).apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=False)

C:\Users\clemm\AppData\Local\Temp\ipykernel_58468\2034372689.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby('site_name', group_keys=False).apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=False)
C:\Users\clemm\AppData\Local\Temp\ipykernel_58468\2034372689.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby('site_name', group_keys=False).apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=False)
C:\Users\clemm\AppData\Local\Temp\ipykernel_58468\2034372689.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.

Power is proportional to the cube of wind speed:

In [22]:
df['wind_speed_100m_cube'] = df['wind_speed_100m'] ** 3

## Feature Engineering: Wind Physics

Wind Shear: Difference in speed between 10m and 100m:

In [23]:
df['wind_shear'] = df['wind_speed_100m'] - df['wind_speed_10m']

## Feature Engineering: Circular Encoding (Direction & Time)

Wind Direction (avoiding the 359° to 0° jump)

In [24]:
df['wind_dir_100m_sin'] = np.sin(np.radians(df['wind_direction_100m']))
df['wind_dir_100m_cos'] = np.cos(np.radians(df['wind_direction_100m']))

Time-based features (Daily and Seasonal cycles)

In [25]:
df['hour'] = df['delivery_time'].dt.hour
df['month'] = df['delivery_time'].dt.month

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 23)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 23)
df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 11)
df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 11)

## Target Normalization (Critical for multi-site modeling)
Predicting Load Factor (0 to 1) instead of raw MW

In [26]:
# df['target'] = df['production'] / df['installed_capacity']

Remove columns that would cause data leakage or are redundant

In [27]:
cols_to_drop = [ 'hour', 'month', 'wind_direction_10m', 'wind_direction_100m'] # 'production',
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

Ajout de la variable target : utilization_rate


In [28]:
df

,index,Unnamed: 0,site_name,delivery_time,production,installed_capacity,utilization_rate,wind_speed_10m,wind_speed_100m,wind_gusts_10m,...,weather_code,sunshine_duration,wind_speed_100m_cube,wind_shear,wind_dir_100m_sin,wind_dir_100m_cos,hour_sin,hour_cos,month_sin,month_cos
0,0,0,Belwind Phase 1,2023-01-01 00:00:00+00:00,147.7025,171.0,86.375731,14.603082,19.897738,20.7,...,51.0,0.0,7877.911982,5.294656,-0.633238,-0.773957,0.000000,1.000000,0.000000,1.000000
1,1,1,Belwind Phase 1,2023-01-01 01:00:00+00:00,146.1775,171.0,85.483918,16.182089,21.681328,20.8,...,51.0,0.0,10191.958316,5.499239,-0.608820,-0.793309,0.269797,0.962917,0.000000,1.000000
2,2,2,Belwind Phase 1,2023-01-01 02:00:00+00:00,146.1800,171.0,85.485380,17.969420,23.809662,24.1,...,51.0,0.0,13497.697496,5.840242,-0.751797,-0.659395,0.519584,0.854419,0.000000,1.000000
3,3,3,Belwind Phase 1,2023-01-01 03:00:00+00:00,146.5050,171.0,85.675439,14.792228,19.860010,23.9,...,3.0,0.0,7833.185089,5.067782,-0.760323,-0.649545,0.730836,0.682553,0.000000,1.000000
4,4,4,Belwind Phase 1,2023-01-01 04:00:00+00:00,146.6950,171.0,85.786550,15.001333,19.915070,19.7,...,3.0,0.0,7898.516174,4.913737,-0.753200,-0.657792,0.887885,0.460065,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274785,274785,274785,Thorntonbank - C-Power - Area SW,2026-02-18 18:00:00+00:00,173.8725,177.6,97.901182,12.904456,15.955643,18.1,...,3.0,0.0,4062.028179,3.051187,0.977711,-0.209957,-0.979084,0.203456,0.540641,0.841254
274786,274786,274786,Thorntonbank - C-Power - Area SW,2026-02-18 19:00:00+00:00,174.1850,177.6,98.077140,12.723305,15.667801,17.9,...,71.0,0.0,3846.131604,2.944496,0.970142,-0.242536,-0.887885,0.460065,0.540641,0.841254
274787,274787,274787,Thorntonbank - C-Power - Area SW,2026-02-18 20:00:00+00:00,160.7725,177.6,90.525056,12.843384,15.946943,17.7,...,3.0,0.0,4055.387197,3.103559,0.968838,-0.247697,-0.730836,0.682553,0.540641,0.841254
274788,274788,274788,Thorntonbank - C-Power - Area SW,2026-02-18 21:00:00+00:00,155.9700,177.6,87.820946,13.947938,17.246014,19.1,...,51.0,0.0,5129.395695,3.298076,0.936448,-0.350807,-0.519584,0.854419,0.540641,0.841254


In [29]:
# Ajout de utilization_rate one hot encoding pour le nom des sites et keep site_name 

df['utilization_rate'] = df['production'] / df['installed_capacity']
site_name_keep = df['site_name']
df = pd.get_dummies(df, columns=['site_name'], prefix='site')

#Add site_name_keep au début du DataFrame
df.insert(0, 'site_name', site_name_keep)

# Ensuite supprimer 'level_1' 
df = df.drop(columns=['level_1', "Unnamed: 0", "index"], errors="ignore")


In [30]:
df.head()

,site_name,delivery_time,production,installed_capacity,utilization_rate,wind_speed_10m,wind_speed_100m,wind_gusts_10m,temperature_2m,dewpoint_2m,...,site_Belwind Phase 1,site_Mermaid Offshore WP,site_Nobelwind Offshore Windpark,site_Norther Offshore WP,site_Northwester 2,site_Northwind,site_Rentel Offshore WP,site_Seastar Offshore WP,site_Thorntonbank - C-Power - Area NE,site_Thorntonbank - C-Power - Area SW
0,Belwind Phase 1,2023-01-01 00:00:00+00:00,147.7025,171.0,0.863757,14.603082,19.897738,20.7,12.25,8.85,...,True,False,False,False,False,False,False,False,False,False
1,Belwind Phase 1,2023-01-01 01:00:00+00:00,146.1775,171.0,0.854839,16.182089,21.681328,20.8,12.10,8.80,...,True,False,False,False,False,False,False,False,False,False
2,Belwind Phase 1,2023-01-01 02:00:00+00:00,146.1800,171.0,0.854854,17.969420,23.809662,24.1,11.85,9.50,...,True,False,False,False,False,False,False,False,False,False
3,Belwind Phase 1,2023-01-01 03:00:00+00:00,146.5050,171.0,0.856754,14.792228,19.860010,23.9,11.80,9.85,...,True,False,False,False,False,False,False,False,False,False
4,Belwind Phase 1,2023-01-01 04:00:00+00:00,146.6950,171.0,0.857865,15.001333,19.915070,19.7,11.75,9.30,...,True,False,False,False,False,False,False,False,False,False


In [31]:
df.to_csv("../data/processed_dataset.csv")